In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M15.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7765981484615143, 'n_it': 0.39280034435135935}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
class OptimisationSetup_class(object):
    def __init__(self):
        self.Parameters_lis = [
            RangeParameterConfig(name="s1", parameter_type="float", bounds=(0, 1)),
            RangeParameterConfig(name="s2", parameter_type="float", bounds=(0, 1)),
            RangeParameterConfig(name="b1", parameter_type="float", bounds=(0, 1)),
        ]
OptimisationSetup_obj = OptimisationSetup_class()

In [5]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [6]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler_obj = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler_obj.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(21): # Run 21 rounds of trials
        trial = sampler_obj.McIntersiteProjTh.McIntersiteProjTh_func(OptimisationSetup_obj,client)
        s1 = trial[0][0]
        s2 = trial[0][1]
        b1 = trial[0][2]
        parameters = {"s1":s1,"s2":s2,"b1":b1}
        trial_index = client.attach_trial(parameters=parameters)
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        result = SurrogateModelOfReality(n_ci,n_it)
        raw_data = {metric_name: result}
        client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    client._experiment.trials.pop(28)
    client._experiment.trials.pop(27)
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
17.55720275094716

Trial 1 =========================================
17.473239746267502

Trial 2 =========================================
15.525512776478465

Trial 3 =========================================
13.552870548219246

Trial 4 =========================================
13.584595420942893

Trial 5 =========================================
14.147371449409693

Trial 6 =========================================
14.531527102468726

Trial 7 =========================================
16.817598755124227

Trial 8 =========================================
13.652105721700234

Trial 9 =========================================
13.332064790588436

Trial 10 =========================================
13.339101397820471

Trial 11 =========================================
13.383491267614994

Trial 12 =========================================
13.384044993797604

Trial 13 =========================================
13.191170175438398

Trial 14 ========

In [7]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.13030713958283
Avg = 14.711386103161534
Std = 1.3453689127872193


In [8]:
print(y_max_arr.tolist())

[17.55720275094716, 17.473239746267502, 15.525512776478465, 13.552870548219246, 13.584595420942893, 14.147371449409693, 14.531527102468726, 16.817598755124227, 13.652105721700234, 13.332064790588436, 13.339101397820471, 13.383491267614994, 13.384044993797604, 13.191170175438398, 16.563583161731607, 13.706479386830987, 14.074733368964727, 13.734150945029642, 13.766679696528508, 17.498820778984413, 16.37036483349069, 15.941067744536195, 16.76025613030939, 14.145059067467104, 17.01153626986274, 18.13030713958283, 13.368446223164744, 15.62391844686, 14.625663789185817, 14.439104192270829, 15.751126643032357, 16.429531431542582, 16.427613335672216, 16.111713732696042, 15.39232704822252, 13.717147913056827, 15.211585275104634, 13.857013612911212, 16.678867047867506, 17.20728276047212, 13.438694934405127, 13.775182417733005, 15.968070793169012, 14.827120413885014, 13.462127539571819, 15.703913897014644, 14.18117491236737, 17.805744189199775, 13.341904724017382, 15.022403314818513, 15.15033001

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M15/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M15/DataGenerated/normal_MIPT_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    13.939566
1    13.562363
2    13.358317
3    15.866150
4    13.781356
..         ...
995  15.312255
996  13.764671
997  13.622958
998  17.414328
999  13.560507

[1000 rows x 1 columns]


In [11]:
# # Sanity check to make sure the MIPT is running correctly.
# df = client.summarize()
# types_lis = []
# for i in range(len(df)):
#     if i < 8:
#         types_lis.append("one-shot")
#     else:
#         types_lis.append("sequential")
# df["type"] = types_lis
# fig = px.scatter_3d(df, x='s1', y='s2', z='b1', color='type',width=1300, height=600)
# fig.show()